In [1]:
import os
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

print("Seeds set for reproducibility")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")


Seeds set for reproducibility
CUDA available: True
CUDA device: NVIDIA A10G


In [2]:
import json
from datetime import datetime

class MultiTaskConfig:
    def __init__(self):
        self.base_model = "microsoft/deberta-v3-base"
        self.goemotions_labels = 28
        self.tweeteval_labels = 4
        self.max_length = 284

        self.hidden_dropout = 0.3
        self.attention_dropout = 0.15
        self.batch_size = 12
        self.learning_rate = 8e-6
        self.num_epochs = 8
        self.warmup_ratio = 0.15
        self.weight_decay = 0.02
        self.gradient_clip = 1.0

        self.goemotions_threshold = 0.35
        self.focal_alpha = 0.3
        self.focal_gamma = 2.5

        self.mixed_precision = True
        self.persistent_workers = True

        self.model_save_path = "./models/multitask"
        self.results_path = "./results/multitask"
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.num_workers = 8 if torch.cuda.is_available() else 0

        os.makedirs(self.model_save_path, exist_ok=True)
        os.makedirs(self.results_path, exist_ok=True)

        self.save_config()

    def save_config(self):
        config_dict = {k: v for k, v in self.__dict__.items()
                      if not k.startswith('device')}
        config_dict['device'] = str(self.device)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        config_path = os.path.join(self.results_path, f"config_{timestamp}.json")

        with open(config_path, 'w') as f:
            json.dump(config_dict, f, indent=2)

        print(f"Configuration saved to {config_path}")



In [3]:
config = MultiTaskConfig()

Configuration saved to ./results/multitask/config_20250916_053728.json


In [4]:
import torch
from torch.utils.data import Dataset

class MultiTaskDataset(Dataset):
    def __init__(self, texts, goemotions_labels, tweeteval_labels, tokenizer, max_length=384, task_type='both'):
        self.texts = texts
        self.goemotions_labels = goemotions_labels if goemotions_labels != [[]] else None
        self.tweeteval_labels = tweeteval_labels if tweeteval_labels != [[]] else None
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.task_type = task_type

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx]).strip()

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        result = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }

        if self.goemotions_labels is not None:
            goemotions_tensor = torch.zeros(28, dtype=torch.float32)
            for label in self.goemotions_labels[idx]:
                goemotions_tensor[label] = 1.0
            result['goemotions_labels'] = goemotions_tensor
        else:
            result['goemotions_labels'] = torch.zeros(28, dtype=torch.float32)

        if self.tweeteval_labels is not None:
            result['tweeteval_labels'] = torch.tensor(self.tweeteval_labels[idx], dtype=torch.long)
        else:
            result['tweeteval_labels'] = torch.tensor(0, dtype=torch.long)

        return result


In [5]:
import torch.nn as nn
from transformers import AutoModel

class MultiTaskModel(nn.Module):
    def __init__(self, config):
        super(MultiTaskModel, self).__init__()
        self.config = config

        self.backbone = AutoModel.from_pretrained(config.base_model)
        hidden_size = self.backbone.config.hidden_size

        self.attention_pooling = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=12,
            dropout=config.attention_dropout,
            batch_first=True
        )

        self.shared_layer = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(config.hidden_dropout)
        )

        self.goemotions_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Dropout(config.hidden_dropout),
            nn.Linear(hidden_size // 2, config.goemotions_labels)
        )

        self.tweeteval_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Dropout(config.hidden_dropout),
            nn.Linear(hidden_size // 2, config.tweeteval_labels)
        )

        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout)

        self._init_weights()

    def _init_weights(self):
        for module in [self.shared_layer, self.goemotions_head, self.tweeteval_head]:
            for layer in module:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_normal_(layer.weight)
                    nn.init.constant_(layer.bias, 0)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state

        cls_output = hidden_states[:, 0, :]

        attended_output, _ = self.attention_pooling(hidden_states, hidden_states, hidden_states)
        mean_attended = attended_output.mean(dim=1)

        combined = torch.cat([cls_output, mean_attended], dim=1)
        shared_features = self.shared_layer(combined)

        goemotions_logits = self.goemotions_head(shared_features)
        tweeteval_logits = self.tweeteval_head(shared_features)

        return goemotions_logits, tweeteval_logits


In [6]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)

        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)

        focal_loss = alpha_t * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [7]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        confidence = 1. - self.smoothing
        logprobs = F.log_softmax(pred, dim=-1)
        nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1))
        nll_loss = nll_loss.squeeze(1)
        smooth_loss = -logprobs.mean(dim=-1)
        loss = confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()

In [8]:
import logging
import json
import csv
import pandas as pd
from datetime import datetime
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup, AutoTokenizer
from sklearn.metrics import accuracy_score, hamming_loss, jaccard_score, precision_recall_fscore_support
from tqdm import tqdm
from transformers import AutoTokenizer
import shutil
import tempfile


In [9]:
import time
from torch.utils.data import DataLoader

In [10]:
class MultiTaskTrainer:
    def __init__(self, config):
        self.config = config
        self.logger = self._setup_logger()
        self.tokenizer = None
        self.model = None
        self.optimizer = None
        self.scheduler = None
        self.scaler = torch.cuda.amp.GradScaler() if config.mixed_precision else None
        self.goemotions_criterion = None
        self.tweeteval_criterion = None
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'val_goemotions_accuracy': [],
            'val_goemotions_precision_macro': [],
            'val_goemotions_precision_micro': [],
            'val_goemotions_recall_macro': [],
            'val_goemotions_recall_micro': [],
            'val_goemotions_f1_macro': [],
            'val_goemotions_f1_micro': [],
            'val_tweeteval_accuracy': [],
            'val_tweeteval_precision_macro': [],
            'val_tweeteval_precision_micro': [],
            'val_tweeteval_recall_macro': [],
            'val_tweeteval_recall_micro': [],
            'val_tweeteval_f1_macro': [],
            'val_tweeteval_f1_micro': [],
            'learning_rates': [],
            'epoch_times': []
        }
        self.results_file = None
        self.detailed_results = []


    def _setup_logger(self):
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(os.path.join(self.config.results_path, 'training.log')),
                logging.StreamHandler()
            ]
        )
        return logging.getLogger(__name__)

    def _save_metrics_to_csv(self, epoch, train_loss, val_metrics, lr, epoch_time):
        
        if self.results_file is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            self.results_file = os.path.join(self.config.results_path, f"training_metrics_{timestamp}.csv")
            with open(self.results_file, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([
                    'epoch', 'train_loss', 'val_loss', 'learning_rate', 'epoch_time',
                    'ge_accuracy', 'ge_precision_macro', 'ge_precision_micro', 'ge_recall_macro', 'ge_recall_micro', 'ge_f1_macro', 'ge_f1_micro',
                    'te_accuracy', 'te_precision_macro', 'te_precision_micro', 'te_recall_macro', 'te_recall_micro', 'te_f1_macro', 'te_f1_micro'
                ])
        
        with open(self.results_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch + 1, train_loss, val_metrics['loss'], lr, epoch_time,
                val_metrics['goemotions_accuracy'], val_metrics['goemotions_precision_macro'], val_metrics['goemotions_precision_micro'],
                val_metrics['goemotions_recall_macro'], val_metrics['goemotions_recall_micro'], val_metrics['goemotions_f1_macro'], val_metrics['goemotions_f1_micro'],
                val_metrics['tweeteval_accuracy'], val_metrics['tweeteval_precision_macro'], val_metrics['tweeteval_precision_micro'],
                val_metrics['tweeteval_recall_macro'], val_metrics['tweeteval_recall_micro'], val_metrics['tweeteval_f1_macro'], val_metrics['tweeteval_f1_micro']
            ])


    def _save_final_results(self):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        results_path = os.path.join(self.config.results_path, f"final_results_{timestamp}.json")
        final_results = {
            'training_history': self.history,
            'best_metrics': {
                'best_goemotions_f1_macro': max(self.history['val_goemotions_f1_macro']),
                'best_goemotions_f1_micro': max(self.history['val_goemotions_f1_micro']),
                'best_tweeteval_accuracy': max(self.history['val_tweeteval_accuracy']),
                'best_tweeteval_f1_macro': max(self.history['val_tweeteval_f1_macro']),
            },
            'config': self.config.__dict__
        }
        with open(results_path, 'w') as f:
            json.dump(final_results, f, indent=2, default=str)
        self.logger.info(f"Final results saved to {results_path}")

    def load_data(self, goemotions_data, tweeteval_data):
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.base_model)
        def create_dataset(dataset, labels_key, is_multilabel=False):
            splits = {}
            for split in ['train', 'validation', 'test']:
                texts = [dataset[split][i]['text'] for i in range(len(dataset[split]))]
                if is_multilabel:
                    labels = [dataset[split][i][labels_key] for i in range(len(dataset[split]))]
                else:
                    labels = [dataset[split][i][labels_key] for i in range(len(dataset[split]))]
                splits[split] = MultiTaskDataset(
                    texts,
                    labels if is_multilabel else [[]],
                    labels if not is_multilabel else [[]],
                    self.tokenizer,
                    self.config.max_length
                )
            return splits['train'], splits['validation'], splits['test']
        ge_train, ge_val, ge_test = create_dataset(goemotions_data, 'labels', is_multilabel=True)
        te_train, te_val, te_test = create_dataset(tweeteval_data, 'label', is_multilabel=False)
        return (ge_train, ge_val, ge_test), (te_train, te_val, te_test)

    
 
    def setup_model(self):
        self.model = MultiTaskModel(self.config).to(self.config.device)
        backbone_params = list(self.model.backbone.parameters())
        head_params = list(self.model.shared_layer.parameters()) + \
                     list(self.model.goemotions_head.parameters()) + \
                     list(self.model.tweeteval_head.parameters()) + \
                     list(self.model.attention_pooling.parameters())
        self.optimizer = AdamW([
            {'params': backbone_params, 'lr': self.config.learning_rate},
            {'params': head_params, 'lr': self.config.learning_rate * 2}
        ], weight_decay=self.config.weight_decay, eps=1e-8)
        self.goemotions_criterion = FocalLoss(
            alpha=self.config.focal_alpha,
            gamma=self.config.focal_gamma
        )
        self.tweeteval_criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
        self.logger.info("Model setup completed")


    def evaluate_single_task(self, dataloader, task_type):
        self.model.eval()
        total_loss = 0
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].to(self.config.device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(self.config.device, non_blocking=True)
                
                if self.config.mixed_precision:
                    with torch.cuda.amp.autocast():
                        goemotions_logits, tweeteval_logits = self.model(input_ids, attention_mask)
                else:
                    goemotions_logits, tweeteval_logits = self.model(input_ids, attention_mask)
                
                if task_type == 'goemotions':
                    labels = batch['goemotions_labels'].to(self.config.device, non_blocking=True)
                    loss = self.goemotions_criterion(goemotions_logits, labels)
                    predictions = (torch.sigmoid(goemotions_logits) > self.config.goemotions_threshold).float()
                    
                    all_predictions.append(predictions.cpu().numpy())
                    all_labels.append(labels.cpu().numpy())
                    
                else:  
                    labels = batch['tweeteval_labels'].to(self.config.device, non_blocking=True)
                    loss = self.tweeteval_criterion(tweeteval_logits, labels)
                    predictions = torch.argmax(tweeteval_logits, dim=1)
                    
                    all_predictions.append(predictions.cpu().numpy())
                    all_labels.append(labels.cpu().numpy())
                
                total_loss += loss.item()
        
        if task_type == 'goemotions':
            all_predictions = np.vstack(all_predictions)
            all_labels = np.vstack(all_labels)
            
            precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
                all_labels, all_predictions, average='macro', zero_division=0
            )
            precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
                all_labels, all_predictions, average='micro', zero_division=0
            )
            
            accuracy = accuracy_score(all_labels, all_predictions)
            
            return {
                'loss': total_loss / len(dataloader),
                'accuracy': accuracy,
                'precision_macro': precision_macro,
                'precision_micro': precision_micro,
                'recall_macro': recall_macro,
                'recall_micro': recall_micro,
                'f1_macro': f1_macro,
                'f1_micro': f1_micro
            }
        else:
            all_predictions = np.concatenate(all_predictions)
            all_labels = np.concatenate(all_labels)
            
            accuracy = accuracy_score(all_labels, all_predictions)
            precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
                all_labels, all_predictions, average='macro', zero_division=0
            )
            precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
                all_labels, all_predictions, average='micro', zero_division=0
            )
            
            return {
                'loss': total_loss / len(dataloader),
                'accuracy': accuracy,
                'precision_macro': precision_macro,
                'precision_micro': precision_micro,
                'recall_macro': recall_macro,
                'recall_micro': recall_micro,
                'f1_macro': f1_macro,
                'f1_micro': f1_micro
            }

    def train(self, ge_datasets, te_datasets):
        ge_train, ge_val, ge_test = ge_datasets
        te_train, te_val, te_test = te_datasets
        
        ge_loader = DataLoader(
            ge_train,
            batch_size=self.config.batch_size,
            shuffle=True,
            num_workers=self.config.num_workers,
            pin_memory=True,
            persistent_workers=self.config.persistent_workers and self.config.num_workers > 0
        )
        te_loader = DataLoader(
            te_train,
            batch_size=self.config.batch_size,
            shuffle=True,
            num_workers=self.config.num_workers,
            pin_memory=True,
            persistent_workers=self.config.persistent_workers and self.config.num_workers > 0
        )
        
        val_ge_loader = DataLoader(
            ge_val,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=self.config.num_workers,
            pin_memory=True
        )
        val_te_loader = DataLoader(
            te_val,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=self.config.num_workers,
            pin_memory=True
        )
        
        self.setup_model()
        total_steps = max(len(ge_loader), len(te_loader)) * self.config.num_epochs
        warmup_steps = int(total_steps * self.config.warmup_ratio)
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
        
        best_combined_score = 0.0
        patience = 3
        no_improve = 0
        
        self.history.update({
            'val_goemotions_accuracy': [],
            'val_goemotions_precision_micro': [],
            'val_goemotions_recall_macro': [],
            'val_goemotions_recall_micro': [],
            'val_tweeteval_precision_micro': [],
            'val_tweeteval_recall_macro': [],
            'val_tweeteval_recall_micro': [],
            'val_tweeteval_f1_micro': []
        })
        
        for epoch in range(self.config.num_epochs):
            epoch_start_time = time.time()
            self.model.train()
            total_loss = 0
            
            ge_iter = iter(ge_loader)
            te_iter = iter(te_loader)
            steps = max(len(ge_loader), len(te_loader))
            
            progress_bar = tqdm(range(steps), desc=f"Epoch {epoch+1}/{self.config.num_epochs}")
            
            for step in progress_bar:
                try:
                    ge_batch = next(ge_iter)
                except StopIteration:
                    ge_iter = iter(ge_loader)
                    ge_batch = next(ge_iter)
                
                try:
                    te_batch = next(te_iter)
                except StopIteration:
                    te_iter = iter(te_loader)
                    te_batch = next(te_iter)
                
                ge_input_ids = ge_batch['input_ids'].to(self.config.device, non_blocking=True)
                ge_attention_mask = ge_batch['attention_mask'].to(self.config.device, non_blocking=True)
                ge_labels = ge_batch['goemotions_labels'].to(self.config.device, non_blocking=True)
                
                te_input_ids = te_batch['input_ids'].to(self.config.device, non_blocking=True)
                te_attention_mask = te_batch['attention_mask'].to(self.config.device, non_blocking=True)
                te_labels = te_batch['tweeteval_labels'].to(self.config.device, non_blocking=True)
                
                self.optimizer.zero_grad()
                
                if self.config.mixed_precision:
                    with torch.cuda.amp.autocast():
                        ge_logits, _ = self.model(ge_input_ids, ge_attention_mask)
                        ge_loss = self.goemotions_criterion(ge_logits, ge_labels)
                        _, te_logits = self.model(te_input_ids, te_attention_mask)
                        te_loss = self.tweeteval_criterion(te_logits, te_labels)
                        loss = ge_loss + te_loss
                    
                    self.scaler.scale(loss).backward()
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.gradient_clip)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    ge_logits, _ = self.model(ge_input_ids, ge_attention_mask)
                    ge_loss = self.goemotions_criterion(ge_logits, ge_labels)
                    _, te_logits = self.model(te_input_ids, te_attention_mask)
                    te_loss = self.tweeteval_criterion(te_logits, te_labels)
                    loss = ge_loss + te_loss
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.gradient_clip)
                    self.optimizer.step()
                
                self.scheduler.step()
                total_loss += loss.item()
                
                progress_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'lr': f'{self.scheduler.get_last_lr()[0]:.2e}'
                })
            
            avg_loss = total_loss / steps
            
            val_ge_metrics = self.evaluate_single_task(val_ge_loader, 'goemotions')
            val_te_metrics = self.evaluate_single_task(val_te_loader, 'tweeteval')
            
            epoch_time = time.time() - epoch_start_time
            current_lr = self.scheduler.get_last_lr()[0]
            
            self.history['train_loss'].append(avg_loss)
            self.history['val_loss'].append((val_ge_metrics['loss'] + val_te_metrics['loss']) / 2)
            
            self.history['val_goemotions_accuracy'].append(val_ge_metrics['accuracy'])
            self.history['val_goemotions_precision_macro'].append(val_ge_metrics['precision_macro'])
            self.history['val_goemotions_precision_micro'].append(val_ge_metrics['precision_micro'])
            self.history['val_goemotions_recall_macro'].append(val_ge_metrics['recall_macro'])
            self.history['val_goemotions_recall_micro'].append(val_ge_metrics['recall_micro'])
            self.history['val_goemotions_f1_macro'].append(val_ge_metrics['f1_macro'])
            self.history['val_goemotions_f1_micro'].append(val_ge_metrics['f1_micro'])
            
            self.history['val_tweeteval_accuracy'].append(val_te_metrics['accuracy'])
            self.history['val_tweeteval_precision_macro'].append(val_te_metrics['precision_macro'])
            self.history['val_tweeteval_precision_micro'].append(val_te_metrics['precision_micro'])
            self.history['val_tweeteval_recall_macro'].append(val_te_metrics['recall_macro'])
            self.history['val_tweeteval_recall_micro'].append(val_te_metrics['recall_micro'])
            self.history['val_tweeteval_f1_macro'].append(val_te_metrics['f1_macro'])
            self.history['val_tweeteval_f1_micro'].append(val_te_metrics['f1_micro'])
            
            self.history['learning_rates'].append(current_lr)
            self.history['epoch_times'].append(epoch_time)
            
            combined_metrics = {
                'loss': (val_ge_metrics['loss'] + val_te_metrics['loss']) / 2,
                'goemotions_accuracy': val_ge_metrics['accuracy'],
                'goemotions_precision_macro': val_ge_metrics['precision_macro'],
                'goemotions_precision_micro': val_ge_metrics['precision_micro'],
                'goemotions_recall_macro': val_ge_metrics['recall_macro'],
                'goemotions_recall_micro': val_ge_metrics['recall_micro'],
                'goemotions_f1_macro': val_ge_metrics['f1_macro'],
                'goemotions_f1_micro': val_ge_metrics['f1_micro'],
                'tweeteval_accuracy': val_te_metrics['accuracy'],
                'tweeteval_precision_macro': val_te_metrics['precision_macro'],
                'tweeteval_precision_micro': val_te_metrics['precision_micro'],
                'tweeteval_recall_macro': val_te_metrics['recall_macro'],
                'tweeteval_recall_micro': val_te_metrics['recall_micro'],
                'tweeteval_f1_macro': val_te_metrics['f1_macro'],
                'tweeteval_f1_micro': val_te_metrics['f1_micro']
            }
            
            self._save_metrics_to_csv(epoch, avg_loss, combined_metrics, current_lr, epoch_time)
            
            print(f"Epoch {epoch+1}/{self.config.num_epochs}")
            print(f"Train Loss: {avg_loss:.4f}")
            print(f"Learning Rate: {current_lr:.2e}")
            print(f"Epoch Time: {epoch_time:.2f}s")
            print("=" * 50)
            
            print("GoEmotions Metrics:")
            print(f"  Accuracy: {val_ge_metrics['accuracy']*100:.2f}%")
            print(f"  Precision Macro: {val_ge_metrics['precision_macro']:.4f}")
            print(f"  Precision Micro: {val_ge_metrics['precision_micro']:.4f}")
            print(f"  Recall Macro: {val_ge_metrics['recall_macro']:.4f}")
            print(f"  Recall Micro: {val_ge_metrics['recall_micro']:.4f}")
            print(f"  F1 Macro: {val_ge_metrics['f1_macro']:.4f}")
            print(f"  F1 Micro: {val_ge_metrics['f1_micro']:.4f}")
            
            print("TweetEval Metrics:")
            print(f"  Accuracy: {val_te_metrics['accuracy']*100:.2f}%")
            print(f"  Precision Macro: {val_te_metrics['precision_macro']:.4f}")
            print(f"  Precision Micro: {val_te_metrics['precision_micro']:.4f}")
            print(f"  Recall Macro: {val_te_metrics['recall_macro']:.4f}")
            print(f"  Recall Micro: {val_te_metrics['recall_micro']:.4f}")
            print(f"  F1 Macro: {val_te_metrics['f1_macro']:.4f}")
            print(f"  F1 Micro: {val_te_metrics['f1_micro']:.4f}")
            print("=" * 50)
            
            combined_score = (val_ge_metrics['f1_macro'] + val_te_metrics['f1_macro']) / 2
            
            if combined_score > best_combined_score:
                best_combined_score = combined_score
                no_improve = 0
                self.save_model('best_model.pt')
                print("Best model saved!")
            else:
                no_improve += 1
                if no_improve >= patience:
                    print("Early stopping triggered!")
                    break
        
        self.save_model('final_model.pt')
        self._save_final_results()


    def predict(self, texts, goemotions_emotion_labels):
        self.model.eval()
        goemotions_results = []
        tweeteval_results = []
        with torch.no_grad():
            for text in tqdm(texts, desc="Predicting"):
                encoding = self.tokenizer(
                    text,
                    truncation=True,
                    padding='max_length',
                    max_length=self.config.max_length,
                    return_tensors='pt'
                ).to(self.config.device)
                if self.config.mixed_precision:
                    with torch.cuda.amp.autocast():
                        goemotions_logits, tweeteval_logits = self.model(
                            encoding['input_ids'],
                            encoding['attention_mask']
                        )
                else:
                    goemotions_logits, tweeteval_logits = self.model(
                        encoding['input_ids'],
                        encoding['attention_mask']
                    )
                ge_probabilities = torch.sigmoid(goemotions_logits).cpu().numpy()[0]
                ge_predictions = {
                    goemotions_emotion_labels[i]: float(prob)
                    for i, prob in enumerate(ge_probabilities)
                }
                te_probabilities = torch.softmax(tweeteval_logits, dim=1).cpu().numpy()[0]
                te_prediction = int(torch.argmax(tweeteval_logits, dim=1).cpu().numpy()[0])
                goemotions_results.append(ge_predictions)
                tweeteval_results.append({
                    'prediction': te_prediction,
                    'probabilities': te_probabilities.tolist()
                })
        return goemotions_results, tweeteval_results
        
    def save_model(self, filename):
        filepath = os.path.join(self.config.model_save_path, filename)
        
        def check_disk_space(path, min_gb=2):
            try:
                total, used, free = shutil.disk_usage(path)
                free_gb = free // (1024**3)
                return free_gb >= min_gb
            except:
                return True  # If we can't check, assume it's okay
        
        if not check_disk_space(self.config.model_save_path):
            self.logger.warning("Low disk space detected. Saving minimal checkpoint.")
            minimal_checkpoint = {
                'model_state_dict': self.model.state_dict(),
                'config': self.config.__dict__,
                'epoch': len(self.history['train_loss'])
            }
            try:
                torch.save(minimal_checkpoint, filepath, _use_new_zipfile_serialization=False)
                self.logger.info(f"Minimal checkpoint saved to {filepath}")
                return
            except Exception as e:
                self.logger.error(f"Failed to save minimal checkpoint: {e}")
                return
        
        checkpoint = {
            'model_state_dict': self.model.state_dict(),
            'config': self.config.__dict__,
            'history': self.history,
            'epoch': len(self.history['train_loss']),
            'best_combined_score': max([
                (self.history['val_goemotions_f1_macro'][i] + self.history['val_tweeteval_f1_macro'][i]) / 2
                for i in range(len(self.history['val_goemotions_f1_macro']))
            ]) if self.history['val_goemotions_f1_macro'] else 0
        }
        
        try:
            if self.optimizer:
                checkpoint['optimizer_state_dict'] = self.optimizer.state_dict()
            if self.scheduler:
                checkpoint['scheduler_state_dict'] = self.scheduler.state_dict()
            if self.scaler:
                checkpoint['scaler_state_dict'] = self.scaler.state_dict()
        except Exception as e:
            self.logger.warning(f"Could not save optimizer/scheduler/scaler states: {e}")
        
        save_methods = [
            lambda: torch.save(checkpoint, filepath, _use_new_zipfile_serialization=False),
            lambda: self._save_with_temp_file(checkpoint, filepath),
            lambda: self._save_components_separately(checkpoint, filepath)
        ]
        
        for i, save_method in enumerate(save_methods):
            try:
                save_method()
                self.tokenizer.save_pretrained(self.config.model_save_path)
                self.logger.info(f"Model checkpoint saved to {filepath} using method {i+1}")
                return
            except Exception as e:
                self.logger.warning(f"Save method {i+1} failed: {e}")
                if i == len(save_methods) - 1:  # Last method failed
                    self.logger.error("All save methods failed. Model not saved.")
                continue

    def load_model(self, filepath, load_optimizer=False, load_scheduler=False):
        self.logger.info(f"Loading model from {filepath}")
        
        try:
            checkpoint = torch.load(filepath, map_location=self.config.device)
            
            if 'model_file' in checkpoint and 'metadata_file' in checkpoint:
                model_checkpoint = torch.load(checkpoint['model_file'], map_location=self.config.device)
                with open(checkpoint['metadata_file'], 'rb') as f:
                    metadata = pickle.load(f)
                checkpoint = {**model_checkpoint, **metadata}
        
            self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_save_path)
            
            if self.model is None:
                self.setup_model()
            
            self.model.load_state_dict(checkpoint['model_state_dict'])
            
            if load_optimizer and 'optimizer_state_dict' in checkpoint:
                if self.optimizer is None:
                    self.setup_model()
                try:
                    self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                except Exception as e:
                    self.logger.warning(f"Could not load optimizer state: {e}")
            
            if load_scheduler and 'scheduler_state_dict' in checkpoint:
                try:
                    if self.scheduler is not None:
                        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                except Exception as e:
                    self.logger.warning(f"Could not load scheduler state: {e}")
            
            if self.scaler and 'scaler_state_dict' in checkpoint:
                try:
                    self.scaler.load_state_dict(checkpoint['scaler_state_dict'])
                except Exception as e:
                    self.logger.warning(f"Could not load scaler state: {e}")
            
            if 'history' in checkpoint:
                self.history = checkpoint['history']
            
            self.logger.info(f"Model loaded successfully from epoch {checkpoint.get('epoch', 'unknown')}")
            return checkpoint
            
        except Exception as e:
            self.logger.error(f"Failed to load model: {e}")
            raise

    

    def _save_with_temp_file(self, checkpoint, filepath):
        import tempfile
        
        temp_dir = os.path.dirname(filepath)
        with tempfile.NamedTemporaryFile(delete=False, dir=temp_dir, suffix='.pt') as tmp_file:
            torch.save(checkpoint, tmp_file.name, _use_new_zipfile_serialization=False)
            os.replace(tmp_file.name, filepath)

    def _save_components_separately(self, checkpoint, filepath):
        base_path = os.path.splitext(filepath)[0]
        
        torch.save({'model_state_dict': checkpoint['model_state_dict']}, 
                   f"{base_path}_model.pt", _use_new_zipfile_serialization=False)
        
        import pickle
        other_components = {k: v for k, v in checkpoint.items() if k != 'model_state_dict'}
        with open(f"{base_path}_metadata.pkl", 'wb') as f:
            pickle.dump(other_components, f)
        
        index = {
            'model_file': f"{base_path}_model.pt",
            'metadata_file': f"{base_path}_metadata.pkl",
            'components': list(other_components.keys())
        }
        torch.save(index, filepath, _use_new_zipfile_serialization=False)

In [11]:
from datasets import load_dataset

def main():
    from datasets import load_dataset

    goemotions_dataset = load_dataset('go_emotions')
    tweeteval_dataset = load_dataset('tweet_eval', 'emotion')

    config = MultiTaskConfig()
    trainer = MultiTaskTrainer(config)

    ge_datasets, te_datasets = trainer.load_data(goemotions_dataset, tweeteval_dataset)
    ge_train, ge_val, ge_test = ge_datasets
    te_train, te_val, te_test = te_datasets

    trainer.train(ge_datasets, te_datasets)

    trainer.load_model(os.path.join(config.model_save_path, 'best_model.pt'))

    ge_test_metrics = trainer.evaluate_single_task(
        DataLoader(ge_test, batch_size=config.batch_size, shuffle=False),
        'goemotions'
    )
    te_test_metrics = trainer.evaluate_single_task(
        DataLoader(te_test, batch_size=config.batch_size, shuffle=False),
        'tweeteval'
    )

    sample_texts = ["I'm so happy today!", "This is really frustrating..."]
    goemotions_labels = [f"emotion_{i}" for i in range(28)]

    ge_predictions, te_predictions = trainer.predict(sample_texts, goemotions_labels)

In [ ]:
if __name__ == "__main__":
    main()


No config specified, defaulting to: go_emotions/simplified
Reusing dataset go_emotions (/home/sagemaker-user/.cache/huggingface/datasets/go_emotions/simplified/0.0.0/2637cfdd4e64d30249c3ed2150fa2b9d279766bfcd6a809b9f085c61a90d776d)


  0%|          | 0/3 [00:00<?, ?it/s]

Reusing dataset tweet_eval (/home/sagemaker-user/.cache/huggingface/datasets/tweet_eval/emotion/1.1.0/12aee5282b8784f3e95459466db4cdf45c6bf49719c25cdb0743d71ed0410343)


  0%|          | 0/3 [00:00<?, ?it/s]

/tmp/ipykernel_1263/1894859659.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler() if config.mixed_precision else None


...

Epoch 8/8:   0%|          | 0/3618 [00:00<?, ?it/s]/tmp/ipykernel_1263/1894859659.py:307: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 8/8: 100%|██████████| 3618/3618 [18:07<00:00,  3.33it/s, loss=0.3609, lr=0.00e+00]
/tmp/ipykernel_1263/1894859659.py:143: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_1263/1894859659.py:143: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 8/8
Train Loss: 0.3591
Learning Rate: 0.00e+00
Epoch Time: 1109.77s
GoEmotions Metrics:
  Accuracy: 39.94%
  Precision Macro: 0.4224
  Precision Micro: 0.5469
  Recall Macro: 0.4131
  Recall Micro: 0.6017
  F1 Macro: 0.3901
  F1 Micro: 0.5730
TweetEval Metrics:
  Accuracy: 82.35%
  Precision Macro: 0.7629
  Precision Micro: 0.8235
  Recall Macro: 0.7646
  Recall Micro: 0.8235
  F1 Macro: 0.7623
  F1 Micro: 0.8235


2025-09-16 08:08:15,333 - __main__ - INFO - Model checkpoint saved to ./models/multitask/best_model.pt using method 1


Best model saved!


2025-09-16 08:08:17,357 - __main__ - INFO - Model checkpoint saved to ./models/multitask/final_model.pt using method 1
2025-09-16 08:08:17,358 - __main__ - INFO - Final results saved to ./results/multitask/final_results_20250916_080817.json
2025-09-16 08:08:17,483 - __main__ - INFO - Loading model from ./models/multitask/best_model.pt
2025-09-16 08:08:19,182 - __main__ - INFO - Model loaded successfully from epoch 8
/tmp/ipykernel_1263/1894859659.py:143: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_1263/1894859659.py:143: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Predicting:   0%|          | 0/2 [00:00<?, ?it/s]/tmp/ipykernel_1263/1894859659.py:443: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cud

In [20]:
def load_and_test_model():
    config = MultiTaskConfig()
    trainer = MultiTaskTrainer(config)
    model_path = os.path.join(config.model_save_path, 'best_model.pt')

    try:
        checkpoint = torch.load(model_path, map_location=config.device)
        trainer.tokenizer = AutoTokenizer.from_pretrained(config.model_save_path)
        trainer.setup_model()
        trainer.model.load_state_dict(checkpoint['model_state_dict'])
        trainer.model.eval()

        print("Model loaded successfully!")
        print(f"Model trained for {checkpoint.get('epoch', 'unknown')} epochs")

        test_texts = [
            "I'm so happy and excited about this!",
            "This is really frustrating and annoying.",
            "What a beautiful day it is today!",
            "I can't believe this happened to me.",
            "Thank you so much for your help!",
            "I'm feeling quite sad about this situation.",
            "This is absolutely amazing and wonderful!",
            "I'm worried about the future."
        ]

        goemotions_labels = [
            'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
            'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment',
            'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness',
            'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'
        ]
        tweeteval_labels = ['joy', 'optimism', 'anger', 'sadness']

        ge_predictions, te_predictions = trainer.predict(test_texts, goemotions_labels)

        for i, text in enumerate(test_texts):
            print(f"\nText {i+1}: {text}")
            print("-" * 60)
            ge_result = ge_predictions[i]
            ge_main_emotion, ge_main_prob = max(ge_result.items(), key=lambda x: x[1])
            print(f"GoEmotions FINAL EMOTION: {ge_main_emotion} (confidence: {ge_main_prob:.4f})")
            ge_sorted = sorted(ge_result.items(), key=lambda x: x[1], reverse=True)[:3]
            print("GoEmotions (Multi-label) - Top 3:")
            for emotion, prob in ge_sorted:
                print(f"  {emotion}: {prob:.4f}")

            te_pred = te_predictions[i]
            te_main_idx = te_pred['prediction']
            te_main_emotion = tweeteval_labels[te_main_idx]
            te_main_confidence = te_pred['probabilities'][te_main_idx]
            print(f"\nTweetEval FINAL EMOTION: {te_main_emotion} (confidence: {te_main_confidence:.4f})")
            
            print("All TweetEval probabilities:")
            for j, (label, prob) in enumerate(zip(tweeteval_labels, te_pred['probabilities'])):
                print(f"  {label}: {prob:.4f}")

    except FileNotFoundError:
        print(f"Model file not found at {model_path}")
        return
    except Exception as e:
        print(f"Error loading model: {e}")
        return



In [21]:
load_and_test_model()


Configuration saved to ./results/multitask/config_20250916_082032.json


/tmp/ipykernel_1263/1894859659.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler() if config.mixed_precision else None
2025-09-16 08:20:34,722 - __main__ - INFO - Model setup completed


Model loaded successfully!
Model trained for 8 epochs


Predicting:   0%|          | 0/8 [00:00<?, ?it/s]/tmp/ipykernel_1263/1894859659.py:443: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Predicting: 100%|██████████| 8/8 [00:00<00:00, 55.36it/s]



Text 1: I'm so happy and excited about this!
------------------------------------------------------------
GoEmotions FINAL EMOTION: joy (confidence: 0.5483)
GoEmotions (Multi-label) - Top 3:
  joy: 0.5483
  excitement: 0.4963
  admiration: 0.2698

TweetEval FINAL EMOTION: optimism (confidence: 0.9380)
All TweetEval probabilities:
  joy: 0.0194
  optimism: 0.9380
  anger: 0.0227
  sadness: 0.0198

Text 2: This is really frustrating and annoying.
------------------------------------------------------------
GoEmotions FINAL EMOTION: disgust (confidence: 0.3955)
GoEmotions (Multi-label) - Top 3:
  disgust: 0.3955
  annoyance: 0.3933
  anger: 0.3813

TweetEval FINAL EMOTION: joy (confidence: 0.9238)
All TweetEval probabilities:
  joy: 0.9238
  optimism: 0.0262
  anger: 0.0228
  sadness: 0.0274

Text 3: What a beautiful day it is today!
------------------------------------------------------------
GoEmotions FINAL EMOTION: admiration (confidence: 0.6948)
GoEmotions (Multi-label) - Top 3:
  a